# Compare the new synthetic instance families

Add three stochastic families: strongly correlated, weakly correlated, and similar-ratio instances. This notebook checks whether they create meaningfully different behavior before we add deliberately adversarial instances.

In [1]:
import numpy as np
import pandas as pd

from knapsack_ml_experiment import (
    generate_similar_ratio_instances,
    generate_strongly_correlated_instances,
    generate_weakly_correlated_instances,
    solve_dynamic_programming,
    solve_greedy,
    split_dataset_by_seed,
)

### Generate and evaluate 1,000 instances per family

The same seed range and number of items are used for every family. This makes the comparison controlled, although a seed does not produce the same weights across generators because they consume random numbers differently.

In [2]:
batches = (
    generate_strongly_correlated_instances(),
    generate_weakly_correlated_instances(),
    generate_similar_ratio_instances(),
)

rows = []
for batch in batches:
    for instance in batch:
        weights = np.array([item.weight for item in instance.items])
        values = np.array([item.value for item in instance.items])
        ratios = values / weights
        greedy = solve_greedy(instance.items, instance.capacity)
        optimal = solve_dynamic_programming(instance.items, instance.capacity)
        failure = greedy.total_value < optimal.total_value

        rows.append(
            {
                "seed": instance.seed,
                "family": instance.family,
                "failure": failure,
                "relative_gap": (
                    (optimal.total_value - greedy.total_value)
                    / optimal.total_value
                ),
                "weight_value_correlation": np.corrcoef(weights, values)[0, 1],
                "ratio_std": ratios.std(),
                "unused_capacity": instance.capacity - greedy.total_weight,
                "selected_items": len(greedy.selected_items),
            }
        )

family_data = pd.DataFrame(rows)
family_data.head()

,seed,family,failure,relative_gap,weight_value_correlation,ratio_std,unused_capacity,selected_items
0,0,strongly_correlated,True,0.057592,1.0,0.696559,22,13
1,1,strongly_correlated,False,0.000000,1.0,1.077265,0,14
2,2,strongly_correlated,True,0.017766,1.0,0.837370,7,14
3,3,strongly_correlated,True,0.017500,1.0,2.110793,7,13
4,4,strongly_correlated,True,0.047887,1.0,1.120572,17,14


### Compare failure frequency and severity

`failure_rate` measures how often greedy is suboptimal. `mean_gap` averages the relative loss over every instance, whereas `failed_case_mean_gap` measures severity only among failures. Do not confuse these quantities.

In [3]:
summary = family_data.groupby("family").agg(
    failure_rate=("failure", "mean"),
    mean_gap=("relative_gap", "mean"),
    failed_case_mean_gap=(
        "relative_gap", lambda gaps: gaps[gaps > 0].mean()
    ),
    mean_weight_value_correlation=("weight_value_correlation", "mean"),
    mean_ratio_std=("ratio_std", "mean"),
    mean_unused_capacity=("unused_capacity", "mean"),
    mean_selected_items=("selected_items", "mean"),
)
summary.round(4)

,failure_rate,mean_gap,failed_case_mean_gap,mean_weight_value_correlation,mean_ratio_std,mean_unused_capacity,mean_selected_items
family,,,,,,,
similar_ratio,0.804,0.0142,0.0177,0.9983,0.0553,3.879,10.920
strongly_correlated,0.924,0.0429,0.0465,1.0000,1.2938,16.979,13.470
weakly_correlated,0.668,0.0111,0.0166,0.9184,0.8174,3.920,11.366


#### Interpretation

- **Strongly correlated:** Values satisfy `value = weight + 10`. This gives light items a disproportionately high ratio because `value / weight = 1 + 10 / weight`. Greedy therefore favors many light items and often leaves capacity that the remaining heavy items cannot fill. This family has both the highest failure rate and the largest average loss.
- **Similar-ratio:** Ratios contain little ranking information, so greedy frequently chooses the wrong combination. However, the alternatives have similar value per unit of weight, which keeps most losses small.
- **Weakly correlated:** The added value variation produces more useful ratio differences. Greedy fails less often, and its failures are usually small.

The important conclusion is that **failure frequency and failure severity are different**. Later, classifier accuracy alone will not describe the quality of the hybrid solver.

### Inspect one failure from each family

In [4]:
examples = []
for batch in batches:
    for instance in batch:
        greedy = solve_greedy(instance.items, instance.capacity)
        optimal = solve_dynamic_programming(instance.items, instance.capacity)
        if greedy.total_value < optimal.total_value:
            examples.append(
                {
                    "family": instance.family,
                    "seed": instance.seed,
                    "capacity": instance.capacity,
                    "greedy_weight": greedy.total_weight,
                    "greedy_value": greedy.total_value,
                    "optimal_weight": optimal.total_weight,
                    "optimal_value": optimal.total_value,
                    "unused_capacity": instance.capacity - greedy.total_weight,
                }
            )
            break

pd.DataFrame(examples).set_index("family")

,seed,capacity,greedy_weight,greedy_value,optimal_weight,optimal_value,unused_capacity
family,,,,,,,
strongly_correlated,0,252,230,360,252,382,22
weakly_correlated,0,272,267,316,270,320,5
similar_ratio,0,270,269,555,270,557,1


In the displayed strongly correlated example, greedy leaves substantially more capacity unused. The weakly correlated and similar-ratio examples nearly fill the knapsack, but a different combination still achieves a slightly greater value.

### Learning checkpoint

Which should the final hybrid treat as more dangerous: frequent small similar-ratio failures, or larger strongly correlated failures? This decision will influence the classification threshold and the downstream metrics we prioritize.

## Split by generation seed

Before model training, each unique generation seed is assigned to exactly one split. All family rows carrying that seed stay together. This prevents seed leakage and, because every family uses seeds 0 through 999, gives every split the same family proportions. The default allocation is 70% training, 15% validation, and 15% test data.

In [5]:
train_data, validation_data, test_data = split_dataset_by_seed(
    family_data, random_seed=42
)

split_summary = pd.concat(
    {
        "train": train_data["family"].value_counts(),
        "validation": validation_data["family"].value_counts(),
        "test": test_data["family"].value_counts(),
    },
    axis=1,
).T
split_summary

family,strongly_correlated,weakly_correlated,similar_ratio
train,700,700,700
validation,150,150,150
test,150,150,150


In [6]:
train_seeds = set(train_data["seed"])
validation_seeds = set(validation_data["seed"])
test_seeds = set(test_data["seed"])

assert train_seeds.isdisjoint(validation_seeds)
assert train_seeds.isdisjoint(test_seeds)
assert validation_seeds.isdisjoint(test_seeds)
assert len(train_seeds | validation_seeds | test_seeds) == 1_000

{
    "train_seeds": len(train_seeds),
    "validation_seeds": len(validation_seeds),
    "test_seeds": len(test_seeds),
    "seed_overlap": 0,
}

{'train_seeds': 700,
 'validation_seeds': 150,
 'test_seeds': 150,
 'seed_overlap': 0}